# 체크포인터와 스레드

같은 thread_id와 다른 thread_id는 저장된 상태를 어떻게 나누는지 확인합니다.

## 실행 환경 준비

Python 3.12 커널에 이 절에서 사용할 패키지 버전을 고정합니다.

In [1]:
# 실습에 필요한 패키지를 같은 버전으로 설치합니다.
%pip install -q langgraph==1.2.11

Note: you may need to restart the kernel to use updated packages.


## 그래프와 체크포인터 준비

`InMemorySaver`를 연결한 작은 누적 그래프를 만듭니다.

In [2]:
# 체크포인터를 붙인 누적 그래프와 상태 형식을 정의합니다.
from typing_extensions import TypedDict
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph

class State(TypedDict):
    total: int
    events: list[str]

def add(state: State) -> dict:
    value = state["events"][-1]
    return {"total": state["total"] + int(value)}

builder = StateGraph(State)
builder.add_node("add", add)
builder.add_edge(START, "add")
builder.add_edge("add", END)
memory = InMemorySaver()
graph = builder.compile(checkpointer=memory)
print("checkpointer:", type(memory).__name__)

checkpointer: InMemorySaver


## 같은 스레드로 두 번 실행

같은 `thread_id`를 사용하면 두 번째 입력을 기존 상태에 이어서 처리합니다.

In [3]:
# customer-a 스레드에 3과 4를 차례로 넣어 누적 상태를 읽습니다.
config_a = {"configurable": {"thread_id": "customer-a"}}
first = graph.invoke({"total": 0, "events": ["3"]}, config_a)
second = graph.invoke({"events": ["4"]}, config_a)
snapshot = graph.get_state(config_a)
print("first total:", first["total"])
print("second total:", second["total"])
print("saved total:", snapshot.values["total"])
print("next:", snapshot.next)

first total: 3
second total: 7
saved total: 7
next: ()


## 다른 스레드와 기록 비교

입력은 같게 두고 `thread_id`만 바꾸어 상태 격리를 확인합니다.

In [4]:
# thread_id만 customer-b로 바꾸어 저장 상태가 섞이지 않는지 비교합니다.
config_b = {"configurable": {"thread_id": "customer-b"}}
other = graph.invoke({"total": 0, "events": ["4"]}, config_b)
history_a = list(graph.get_state_history(config_a))
history_b = list(graph.get_state_history(config_b))
print("customer-a total:", second["total"], "history:", len(history_a))
print("customer-b total:", other["total"], "history:", len(history_b))
assert second["total"] == 7 and other["total"] == 4

customer-a total: 7 history: 6
customer-b total: 4 history: 3
